In [1]:
import pandas as pd, textstat, json
import plotly_express as px
from readability import Readability

#### Readability

In [40]:
# Import JSON XAIQB answers
with open("xaiqb_human.json") as f:
    xaiqb_human = json.load(f)

with open("xaiqb_gemini_1.1.json") as f:
    xaiqb_gemini = json.load(f)

with open("xaiqb_chatgpt_1.1_v2.json") as f:
    xaiqb_chatgpt = json.load(f)

# with open("xaiqb_chatgpt.json") as f:
#     xaiqb_chatgpt = json.load(f)

# with open("xaiqb_gemini.json") as f:
#     xaiqb_gemini = json.load(f)

# with open("xaiqb_claude.json") as f:
#     xaiqb_claude = json.load(f)

# with open("xaiqb_mistral.json") as f:
#     xaiqb_mistral = json.load(f)

# with open("xaiqb_llama.json") as f:
#     xaiqb_llama = json.load(f)

In [ ]:
# https://readabilityformulas.com/
    # https://pypi.org/project/py-readability-metrics/
    
    # if textstat.textstat.lexicon_count(text) >= 100:
    #     r = Readability(text)

    #     scores = {
    #         "text": text,
    #         # "stats": r.statistics(),
    #         "flesch_kincaid": {
    #             "score": r.flesch_kincaid().score, 
    #             "grade_level": r.flesch_kincaid().grade_level,
    #         },
    #         "flesch_reading": {
    #             "score": r.flesch().score,
    #             "ease": r.flesch().ease, 
    #             "grade_level": r.flesch().grade_levels,
    #         },
    #         "dale_chall": {
    #             "score": r.dale_chall().score,
    #             "grade_level": r.dale_chall().grade_levels,
    #         },
    #         "ari": {
    #             "score": r.ari().score,
    #             "grade_level": r.ari().grade_levels,
    #             "ages": r.ari().ages
    #         },
    #         "coleman_liau": {
    #             "score": r.coleman_liau().score, 
    #             "grade_level": r.coleman_liau().grade_level
    #         },
    #         "gunning_fog": {
    #             "score": r.gunning_fog().score, 
    #             "grade_level": r.gunning_fog().grade_level
    #         },
    #         "smog": {
    #             "score": r.smog().score, 
    #             "grade_level": r.smog().grade_level
    #         },
    #         "spache": {
    #             "score": r.spache().score, 
    #             "grade_level": r.spache().grade_level
    #         },
    #         "linsear_write": {
    #             "score": r.linsear_write().score, 
    #             "grade_level": r.linsear_write().grade_level
    #         }
    #     }
    # else:

# pd.DataFrame(scores).drop(["text"], axis = 1).drop(["ease", "ages"], axis = 0) if df else scores 

In [ ]:
def readabililty_scores(text):
    
    scores = {
        "flesch_reading_ease":          round(textstat.flesch_reading_ease(text), 3),
        "smog_index":                   round(textstat.smog_index(text), 3),
        "coleman_liau_index":           round(textstat.coleman_liau_index(text), 3),
        "dale_chall_readability_score": round(textstat.dale_chall_readability_score(text), 3),
        "text_standard":                textstat.text_standard(text),
        "difficult_words":              round(textstat.difficult_words(text), 3),
        
        # Ignore/irrelevant
        # "flesch_kincaid_grade":         round(textstat.flesch_kincaid_grade(text), 3),
        # "automated_readability_index":  round(textstat.automated_readability_index(text), 3),
        # "linsear_write_formula":        round(textstat.linsear_write_formula(text), 3),
        # "gunning_fog":                  round(textstat.gunning_fog(text), 3),
        # "spache_readability":           round(textstat.spache_readability(text), 3)
        }

    return scores

In [47]:
# Define function to calculate readability scores for all questions
def get_scores(x = xaiqb_human):
    scores_dict = {}
    for category in x.keys():
        scores_dict.update({category: {}})
        for question, answer in x[category]["question_answer"].items():
            scores = readabililty_scores(answer["answer"])
            scores_dict[category].update({question: scores})
    return scores_dict

In [ ]:
scores_human = get_scores(x = xaiqb_human)
scores_chatgpt = get_scores(x = xaiqb_chatgpt)
scores_gemini = get_scores(x = xaiqb_gemini)

In [172]:
def pull_scores(x = scores_human, category = "Data", metric = "flesch_reading_ease"):
    scores = []
    for score in x[category].values():
        scores += [score[metric]]
    return scores

In [482]:
pd.DataFrame(scores_human["Data"].values()).iloc[[0]]

,flesch_reading_ease,flesch_kincaid_grade,smog_index,coleman_liau_index,automated_readability_index,dale_chall_readability_score,linsear_write_formula,gunning_fog,text_standard,spache_readability
0,84.9,3.653333,3.1291,7.977778,5.835,11.100678,3.5,3.6,3rd and 4th grade,4.496889


In [281]:
to_plot = []
for category in scores_human.keys():
    for metric in [
        # # "flesch_reading_ease", 
        "flesch_kincaid_grade",
        "smog_index",
        # "coleman_liau_index",
        # "automated_readability_index",
        # "dale_chall_readability_score",
        # "linsear_write_formula",
        "gunning_fog", 
        # # "text_standard",
        # "spache_readability"
        ]:
        to_plot += [pd.DataFrame({
            "category": category,
            "metric": metric,
            "human": pull_scores(x = scores_human, category = category, metric = metric),
            "chatgpt": pull_scores(x = scores_chatgpt, category = category, metric = metric),
            "gemini": pull_scores(x = scores_gemini, category = category, metric = metric),
            "claude": pull_scores(x = scores_claude, category = category, metric = metric),
            "mistral": pull_scores(x = scores_mistral, category = category, metric = metric),
            "llama": pull_scores(x = scores_llama, category = category, metric = metric)
        })]

to_plot = pd.concat(to_plot).melt(id_vars = ["category", "metric"], var_name = "model", value_name = "score")

# fig = px.box(to_plot[to_plot["metric"] == "flesch_reading_ease"], x = "score", color = "model", points = "all", 
#              facet_row = "category", facet_col = "metric", 
#              height = 3000, width = 1300
#              ).update_xaxes(matches = None)

# fig.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels = True, title_font = dict(size = 20)))
# fig.show()

In [478]:
# Facet by category and metric
fig = px.box(to_plot[to_plot["category"].isin(['How (global model-wide explanation)', 'Performance'])], 
             x = "score", color = "model", points = "all", 
             facet_row = "category", facet_col = "metric", 
             height = 1000, width = 1300,
             title = "Readability Metrics by Category: Human vs LLM Output"
             ).update_xaxes(matches = None).update_layout(title_x = 0.5)

fig.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels = True, title_font = dict(size = 20)))
fig.show()

In [473]:
# Facet only by metric
fig = px.violin(to_plot, x = "score", color = "model", 
                box = True, points = "all",
                facet_col = "metric", 
                height = 750, width = 1300,
                title = "Overall Readability Metrics: Human vs LLM Output"
                ).update_xaxes(matches = None).update_layout(title_x = 0.5)

# fig = px.box(to_plot, x = "score", color = "model", 
#              points = "all",
#              facet_col = "metric", 
#              height = 500, width = 1300
#              ).update_xaxes(matches = None).update_layout(boxmode = "group", boxgap = 0, boxgroupgap = 0.5)

fig.show()

##### Similarity

https://spotintelligence.com/2022/12/19/text-similarity-python/#1_Text_similarity_with_NLTK

In [336]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [381]:
import numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [493]:
similarity_scores = []
for category in xaiqb_human["Global"].keys():    
    for question in xaiqb_human["Global"][category]["question_answer"].keys():
        t1 = xaiqb_human["Global"][category]["question_answer"][question]
        t2 = xaiqb_gemini["Global"][category]["question_answer"][question]
        # Tokenize and encode the texts
        encoding1 = model.encode(t1, normalize_embeddings = True)
        encoding2 = model.encode(t2, normalize_embeddings = True)
        # Calculate the cosine similarity between the embeddings
        similarity = np.dot(encoding1, encoding2) / (np.linalg.norm(encoding1) * np.linalg.norm(encoding2))
        similarity_scores += [pd.DataFrame({
            "category": category,
            "question": question,
            "similarity": [similarity]
        })]

similarity_scores = pd.concat(similarity_scores).sort_values("similarity", ascending = False).reset_index(drop = True)

In [494]:
pd.set_option('display.max_colwidth', None)

In [495]:
similarity_scores

,category,question,similarity
0,How (global model-wide explanation),What are the top rules/features that determine its predictions?,0.825490
1,Data,What is the source of the training data?,0.813104
2,Performance,What kind of mistakes is the system likely to make?,0.799797
3,Output,How should the output fit in my workflow?,0.716793
4,UI,What are the aﬀordances of the system?,0.692760
5,Performance,In what situations is the system likely to be correct/incorrect?,0.673187
6,Output,What does the system output mean?,0.655534
7,How (global model-wide explanation),What kind of algorithm is used?,0.655185
8,Data,How were the labels/ground-truth produced?,0.630064
9,Performance,How accurate/precise/reliable are the predictions?,0.597019


In [448]:
(similarity_scores
 .groupby("category")
 .agg({"similarity": ["mean", "size"]})
 .reset_index()
 .sort_values([("similarity", "mean")], ascending = False)
 .reset_index(drop = True)
)

category similarity     
                                             mean size
0                                   UI   0.595625    2
1  How (global model-wide explanation)   0.547012    8
2                          Performance   0.543399    5
3                               Output   0.417338   10
4                                 Data   0.386063    6
5                               Others   0.374370    4

In [485]:
largest_similarity = similarity_scores.nlargest(3, "similarity").reset_index(drop = True)
largest_similarity

,category,question,similarity
0,How (global model-wide explanation),What are the top rules/features that determine...,0.825490
1,Data,What is the source of the training data?,0.813104
2,Performance,What kind of mistakes is the system likely to ...,0.799797


In [486]:
for category, question, similarity in largest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human["Global"][category]["question_answer"][question]}
    LLM:\t{xaiqb_gemini["Global"][category]["question_answer"][question]}
    ''')


    How (global model-wide explanation) | What are the top rules/features that determine its predictions?
    Similarity score: 0.825
    --------------------------------------------------------------------------------
    Human:	Top features are if the doctor reccomends the H1N1 vaccine, if they believe the vaccine is effective, and how much of a risk they believe H1N1 is.
    LLM:	The top three most influential features are whether a doctor recommended the H1N1 vaccine, the respondent's perceived risk of getting sick from H1N1, and their opinion on the vaccine's effectiveness.【2-34】
    

    Data | What is the source of the training data?
    Similarity score: 0.813
    --------------------------------------------------------------------------------
    Human:	Data comes from the National 2009 H1N1 Flu Survey (NHFS).
    LLM:	The data comes from the 'Flu Shot Learning: Predict H1N1 and Seasonal Flu Vaccines' competition on DrivenData, which is based on the National 2009 H1N1 Flu Su

In [466]:
smallest_similarity = similarity_scores.nsmallest(3, "similarity").reset_index(drop = True)
smallest_similarity

,category,question,similarity
0,Data,What dataset(s) is the system NOT using?,0.097394
1,Data,What are the potential limitations/biases of t...,0.118485
2,Output,What is the scope of the output data?,0.129578


In [467]:
for category, question, similarity in smallest_similarity.values:
    print(f'''
    {category} | {question}
    Similarity score: {similarity:.3f}
    {"-" * 80}
    Human:\t{xaiqb_human["Global"][category]["question_answer"][question]}
    LLM:\t{xaiqb_gemini["Global"][category]["question_answer"][question]}
    ''')


    Data | What dataset(s) is the system NOT using?
    Similarity score: 0.097
    --------------------------------------------------------------------------------
    Human:	Only data provided by these two csv files are being used by the system.
    LLM:	The model is trained specifically on H1N1-related features and excludes features related to the seasonal flu vaccine to focus the prediction.【2-66】
    

    Data | What are the potential limitations/biases of the data?
    Similarity score: 0.118
    --------------------------------------------------------------------------------
    Human:	Limited transferability, as all respondants were persons 6 months or older living in the United States at the time of the interview, over telephone.
    LLM:	The dataset has a significant number of missing values in key fields like health insurance, employment industry, and occupation, which could introduce bias. For example, nearly 46% of respondents have missing health insurance information.【2

#### Readability Metric Descriptions

##### Flesch-Kincaid Grade Level
The U.S. Army uses Flesch-Kincaid Grade Level for assessing the difficulty of technical manuals. The commonwealth of Pennsylvania uses Flesch-Kincaid Grade Level for scoring automobile insurance policies to ensure their texts are no higher than a ninth grade level of reading difficulty. Many other U.S. states also use Flesch-Kincaid Grade Level to score other legal documents such as business policies and financial forms.

##### Flesch Reading Ease
The U.S. Department of Defense uses the Reading Ease test as the standard test of readability for its documents and forms. Florida requires that life insurance policies have a Flesch Reading Ease score of 45 or greater.

##### Dale Chall Readability
The Dale-Chall Formula is an accurate readability formula for the simple reason that it is based on the use of familiar words, rather than syllable or letter counts. Reading tests show that readers usually find it easier to read, process and recall a passage if they find the words familiar.

##### Automated Readability Index (ARI)
Unlike the other indices, the ARI, along with the Coleman-Liau, relies on a factor of characters per word, instead of the usual syllables per word. ARI is widely used on all types of texts.

##### Coleman Liau Index
The Coleman-Liau Formula usually gives a lower grade value than any of the Kincaid, ARI and Flesch values when applied to technical documents.

##### Gunning Fog
The Gunning fog index measures the readability of English writing. The index estimates the years of formal education needed to understand the text on a first reading. A fog index of 12 requires the reading level of a U.S. high school senior (around 18 years old).

##### SMOG
The SMOG Readability Formula (Simple Measure of Gobbledygook) is a popular method to use on health literacy materials.

##### SPACHE
The Spache Readability Formula is used for Primary-Grade Reading Materials, published in 1953 in The Elementary School Journal. The Spache Formula is best used to calculate the difficulty of text that falls at the 3rd grade level or below.

##### Linsear Write
Linsear Write is a readability metric for English text, purportedly developed for the United States Air Force to help them calculate the readability of their technical manuals.